In [2]:
import pandas as pd
import pyodbc
def create_connection():
    conn=pyodbc.connect(
        "DRIVER={ODBC Driver 18 for SQL Server};"
        "SERVER=atwpSQL-STP-app;"
        "DATABASE=LinePC7442;"
        "Trusted_Connection=yes;"
        "TrustServerCertificate=yes;"
    )
    return conn
def get_missing_ids(conn):
    query="""
        SELECT
            [ID],
            [Prog_Nr],
            [ordername],
            [stable_ts_start],
            [stable_ts_stop],
            [productions_runs],
            [prodRun_time],
            [description],
            [line_speed_min],
            [line_speed_max],
            [line_speed_avg],
            [line_speed_std]
        FROM [LinePC7442].[dbo].[Ex2_stableProd_dev]
        WHERE
            [valid]=1
            AND
            (
                [prodRun_time] IS NULL
                OR [description] IS NULL
                OR [min_pressure] IS NULL
                OR [max_pressure] IS NULL
                OR [avg_pressure] IS NULL
                OR [std_pressure] IS NULL
                OR [max_temp] IS NULL
                OR [avg_temp] IS NULL
                OR [std_temp] IS NULL
                OR [rpm_min] IS NULL
                OR [rpm_max] IS NULL
                OR [rpm_avg] IS NULL
                OR [rpm_std] IS NULL
                OR [OD_real_min] IS NULL
                OR [OD_real_max] IS NULL
                OR [OD_real_avg] IS NULL
                OR [OD_real_std] IS NULL
                OR [line_speed_min] IS NULL
                OR [line_speed_max] IS NULL
                OR [line_speed_avg] IS NULL
                OR [line_speed_std] IS NULL
            )
        ORDER BY [ID]
    """
    return pd.read_sql(query,conn)
def get_statistics_data(conn,prog_nr,ordername,stable_start,stable_stop):
    query="""
        SELECT
            d.[timestamp],
            CAST(d.[Prog_Nr] AS VARCHAR(255)) AS [Prog_Nr],
            d.[ordername],
            d.[Extr_ist] AS rpm,
            d.[Auszen_DM_XY_ist] AS OD_real,
            ISNULL(d.[Ausstoz_ist],0) AS Ausstoz_ist,
            d.[Massedruck],
            d.[Mass_ist],
            d.[Linie_ist] AS line_speed
        FROM [LinePC7442].[dbo].[DI_Ex2_Istwerte] d
        WHERE
            d.[timestamp]>=?
            AND d.[timestamp]<=?
            AND CAST(d.[Prog_Nr] AS VARCHAR(255))=?
            AND d.[ordername]=?
        ORDER BY d.[timestamp]
    """
    df=pd.read_sql(
        query,
        conn,
        params=[
            stable_start,
            stable_stop,
            str(prog_nr),
            ordername
        ]
    )
    if df.empty:
        return df
    df["timestamp"]=pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )
    df=df.drop_duplicates(
        subset=[
            "timestamp",
            "Prog_Nr",
            "ordername"
        ],
        keep="first"
    )
    numeric_columns=[
        "rpm",
        "OD_real",
        "Massedruck",
        "Mass_ist",
        "line_speed"
    ]
    for col in numeric_columns:
        df[col]=pd.to_numeric(
            df[col],
            errors="coerce"
        )
    return df
def get_recipe_description(conn,prog_nr):
    query="""
        SELECT TOP 1
            [description]
        FROM [LinePC7442].[dbo].[troester_recipe]
        WHERE CAST([recipename] AS VARCHAR(255))=CAST(? AS VARCHAR(255))
        ORDER BY [timestamp] DESC
    """
    cursor=conn.cursor()
    cursor.execute(
        query,
        (prog_nr,)
    )
    result=cursor.fetchone()
    cursor.close()
    if result is None:
        return None
    return result[0]
def calculate_statistics(df):
    if df.empty:
        return None
    pressure=df["Massedruck"].dropna()
    if len(pressure)>0:
        min_pressure=pressure.min()
        max_pressure=pressure.max()
        avg_pressure=pressure.mean()
        std_pressure=pressure.std()
    else:
        min_pressure=None
        max_pressure=None
        avg_pressure=None
        std_pressure=None
    temperature=df["Mass_ist"].dropna()
    if len(temperature)>0:
        max_temp=temperature.max()
        avg_temp=temperature.mean()
        std_temp=temperature.std()
    else:
        max_temp=None
        avg_temp=None
        std_temp=None
    rpm=df["rpm"].dropna()
    if len(rpm)>0:
        rpm_min=rpm.min()
        rpm_max=rpm.max()
        rpm_avg=rpm.mean()
        rpm_std=rpm.std()
    else:
        rpm_min=None
        rpm_max=None
        rpm_avg=None
        rpm_std=None
    od=df["OD_real"].dropna()
    if len(od)>0:
        od_min=od.min()
        od_max=od.max()
        od_avg=od.mean()
        od_std=od.std()
    else:
        od_min=None
        od_max=None
        od_avg=None
        od_std=None
    line_speed=df["line_speed"].dropna()
    if len(line_speed)>0:
        line_speed_min=line_speed.min()
        line_speed_max=line_speed.max()
        line_speed_avg=line_speed.mean()
        line_speed_std=line_speed.std()
    else:
        line_speed_min=None
        line_speed_max=None
        line_speed_avg=None
        line_speed_std=None
    return{
        "min_pressure":min_pressure,
        "max_pressure":max_pressure,
        "avg_pressure":avg_pressure,
        "std_pressure":std_pressure,
        "max_temp":max_temp,
        "avg_temp":avg_temp,
        "std_temp":std_temp,
        "rpm_min":rpm_min,
        "rpm_max":rpm_max,
        "rpm_avg":rpm_avg,
        "rpm_std":rpm_std,
        "OD_real_min":od_min,
        "OD_real_max":od_max,
        "OD_real_avg":od_avg,
        "OD_real_std":od_std,
        "line_speed_min":line_speed_min,
        "line_speed_max":line_speed_max,
        "line_speed_avg":line_speed_avg,
        "line_speed_std":line_speed_std
    }
def calculate_prod_run_time(stable_start,stable_stop):
    if pd.isna(stable_start) or pd.isna(stable_stop):
        return None
    duration_seconds=(
        pd.Timestamp(stable_stop)-
        pd.Timestamp(stable_start)
    ).total_seconds()
    if duration_seconds<0:
        return None
    return duration_seconds/60.0
def update_statistics(
    conn,
    row_id,
    statistics,
    prod_run_time,
    description
):
    query="""
        UPDATE [LinePC7442].[dbo].[Ex2_stableProd_dev]
        SET
            [min_pressure]=?,
            [max_pressure]=?,
            [avg_pressure]=?,
            [std_pressure]=?,
            [max_temp]=?,
            [avg_temp]=?,
            [std_temp]=?,
            [rpm_min]=?,
            [rpm_max]=?,
            [rpm_avg]=?,
            [rpm_std]=?,
            [OD_real_min]=?,
            [OD_real_max]=?,
            [OD_real_avg]=?,
            [OD_real_std]=?,
            [prodRun_time]=?,
            [description]=?,
            [line_speed_min]=?,
            [line_speed_max]=?,
            [line_speed_avg]=?,
            [line_speed_std]=?
        WHERE [ID]=?
    """
    cursor=conn.cursor()
    cursor.execute(
        query,
        (
            statistics["min_pressure"],
            statistics["max_pressure"],
            statistics["avg_pressure"],
            statistics["std_pressure"],
            statistics["max_temp"],
            statistics["avg_temp"],
            statistics["std_temp"],
            statistics["rpm_min"],
            statistics["rpm_max"],
            statistics["rpm_avg"],
            statistics["rpm_std"],
            statistics["OD_real_min"],
            statistics["OD_real_max"],
            statistics["OD_real_avg"],
            statistics["OD_real_std"],
            prod_run_time,
            description,
            statistics["line_speed_min"],
            statistics["line_speed_max"],
            statistics["line_speed_avg"],
            statistics["line_speed_std"],
            int(row_id)
        )
    )
    cursor.close()
def update_prod_run_time_only(
    conn,
    row_id,
    prod_run_time,
    description
):
    query="""
        UPDATE [LinePC7442].[dbo].[Ex2_stableProd_dev]
        SET
            [prodRun_time]=?,
            [description]=?
        WHERE [ID]=?
    """
    cursor=conn.cursor()
    cursor.execute(
        query,
        (
            float(prod_run_time),
            description,
            int(row_id)
        )
    )
    cursor.close()
def auto_updated_rows(conn=None):
    own_connection=False
    if conn is None:
        conn=create_connection()
        own_connection=True
    try:
        missing_rows=get_missing_ids(conn)
        if missing_rows.empty:
            print(
                "All valid production rows already have "
                "statistics, description, line speed statistics "
                "and prodRun_time."
            )
            return []
        print()
        print(
            f"Rows requiring update: "
            f"{len(missing_rows)}"
        )
        updated=[]
        failed=[]
        for _,row in missing_rows.iterrows():
            row_id=row["ID"]
            prog_nr=row["Prog_Nr"]
            ordername=row["ordername"]
            stable_start=row["stable_ts_start"]
            stable_stop=row["stable_ts_stop"]
            production_run=row["productions_runs"]
            old_prod_run_time=row["prodRun_time"]
            print()
            print("-"*70)
            print(f"ID               : {row_id}")
            print(f"Prog_Nr          : {prog_nr}")
            print(f"Order            : {ordername}")
            print(f"Production Run   : {production_run}")
            print(f"Stable Start     : {stable_start}")
            print(f"Stable Stop      : {stable_stop}")
            print(f"Old prodRun_time : {old_prod_run_time}")
            if pd.isna(stable_start) or pd.isna(stable_stop):
                print(
                    "Skipped: invalid stable timestamps."
                )
                failed.append(int(row_id))
                continue
            try:
                prod_run_time=calculate_prod_run_time(
                    stable_start,
                    stable_stop
                )
                if prod_run_time is None:
                    print(
                        "Skipped: could not calculate "
                        "prodRun_time."
                    )
                    failed.append(int(row_id))
                    continue
                description=get_recipe_description(
                    conn,
                    prog_nr
                )
                print(
                    f"Description      : "
                    f"{description if description is not None else 'Not found'}"
                )
                print(
                    f"Calculated prodRun_time: "
                    f"{prod_run_time:.2f} minutes"
                )
                df=get_statistics_data(
                    conn,
                    prog_nr,
                    ordername,
                    stable_start,
                    stable_stop
                )
                if df.empty:
                    print(
                        "No machine data found "
                        "inside stable window."
                    )
                    update_prod_run_time_only(
                        conn,
                        row_id,
                        prod_run_time,
                        description
                    )
                    updated.append(int(row_id))
                    print(
                        "prodRun_time and description updated, "
                        "statistics not available."
                    )
                    continue
                print(
                    f"Measurement rows: "
                    f"{len(df)}"
                )
                statistics=calculate_statistics(df)
                if statistics is None:
                    update_prod_run_time_only(
                        conn,
                        row_id,
                        prod_run_time,
                        description
                    )
                    updated.append(int(row_id))
                    print(
                        "prodRun_time and description updated, "
                        "statistics could not be calculated."
                    )
                    continue
                print(
                    f"Line speed min   : "
                    f"{statistics['line_speed_min']}"
                )
                print(
                    f"Line speed max   : "
                    f"{statistics['line_speed_max']}"
                )
                print(
                    f"Line speed avg   : "
                    f"{statistics['line_speed_avg']}"
                )
                print(
                    f"Line speed std   : "
                    f"{statistics['line_speed_std']}"
                )
                update_statistics(
                    conn,
                    row_id,
                    statistics,
                    prod_run_time,
                    description
                )
                updated.append(int(row_id))
                print(
                    "Statistics, line speed statistics, "
                    "description and prodRun_time "
                    "updated successfully."
                )
            except Exception as e:
                print(
                    f"Failed ID {row_id}: {e}"
                )
                failed.append(int(row_id))
        conn.commit()
        print()
        print("="*70)
        print("Statistics update completed.")
        print(f"Updated: {len(updated)}")
        print(f"Failed : {len(failed)}")
        print("="*70)
        return updated
    except Exception:
        conn.rollback()
        raise
    finally:
        if own_connection:
            conn.close()
def clean_calc():
    conn=create_connection()
    try:
        cursor=conn.cursor()
        query="""
            UPDATE [LinePC7442].[dbo].[Ex2_stableProd_dev]
            SET
                [min_pressure]=NULL,
                [max_pressure]=NULL,
                [avg_pressure]=NULL,
                [std_pressure]=NULL,
                [max_temp]=NULL,
                [avg_temp]=NULL,
                [std_temp]=NULL,
                [rpm_min]=NULL,
                [rpm_max]=NULL,
                [rpm_avg]=NULL,
                [rpm_std]=NULL,
                [OD_real_min]=NULL,
                [OD_real_max]=NULL,
                [OD_real_avg]=NULL,
                [OD_real_std]=NULL,
                [line_speed_min]=NULL,
                [line_speed_max]=NULL,
                [line_speed_avg]=NULL,
                [line_speed_std]=NULL
        """
        cursor.execute(query)
        conn.commit()
        cursor.close()
        print(
            "All calculated statistics and line speed statistics "
            "have been reset to NULL. prodRun_time, description "
            "and stable_meters were not changed."
        )
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()
if __name__=="__main__":
    auto_updated_rows()


C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:56: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query,conn)



Rows requiring update: 2469

----------------------------------------------------------------------
ID               : 20288
Prog_Nr          : 2133
Order            : 6140
Production Run   : 1
Stable Start     : 2025-01-07 07:48:49
Stable Stop      : 2025-01-07 08:14:45
Old prodRun_time : 25.933333333333334
Description      : 3048_25,0_4,0
Calculated prodRun_time: 25.93 minutes
Measurement rows: 779
Line speed min   : 12.699999809265137
Line speed max   : 14.300000190734863
Line speed avg   : 13.781129715837471
Line speed std   : 0.28209785862508535
Statistics, line speed statistics, description and prodRun_time updated successfully.

----------------------------------------------------------------------
ID               : 20289
Prog_Nr          : 2133
Order            : 6140
Production Run   : 2
Stable Start     : 2025-01-07 10:35:01
Stable Stop      : 2025-01-07 10:52:09
Old prodRun_time : 17.133333333333333
Description      : 3048_25,0_4,0
Calculated prodRun_time: 17.13 minutes


C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(
C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(


Measurement rows: 515
Line speed min   : 12.0
Line speed max   : 15.699999809265137
Line speed avg   : 14.67262134181643
Line speed std   : 1.0467142802319298
Statistics, line speed statistics, description and prodRun_time updated successfully.

----------------------------------------------------------------------
ID               : 20290
Prog_Nr          : 2133
Order            : 6140
Production Run   : 3
Stable Start     : 2025-01-07 11:04:29
Stable Stop      : 2025-01-07 12:16:29
Old prodRun_time : 72.0
Description      : 3048_25,0_4,0
Calculated prodRun_time: 72.00 minutes


C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(


...

C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(
C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(


Measurement rows: 1518
Line speed min   : 10.199999809265137
Line speed max   : 13.100000381469727
Line speed avg   : 12.551646947546596
Line speed std   : 0.4123752709167278
Statistics, line speed statistics, description and prodRun_time updated successfully.

----------------------------------------------------------------------
ID               : 23589
Prog_Nr          : 9021 /4405
Order            : 6277
Production Run   : 1
Stable Start     : 2026-09-17 14:20:03
Stable Stop      : 2026-09-17 14:48:05
Old prodRun_time : 28.033333333333335
Description      : HD DN38,0xSeele 45,2
Calculated prodRun_time: 28.03 minutes
Measurement rows: 842
Line speed min   : 8.199999809265137
Line speed max   : 14.199999809265137
Line speed avg   : 12.450593750154038
Line speed std   : 1.9623218618337628
Statistics, line speed statistics, description and prodRun_time updated successfully.

----------------------------------------------------------------------
ID               : 23591
Prog_Nr         

C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(
C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(
C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(


Measurement rows: 464
Line speed min   : 8.399999618530273
Line speed max   : 9.100000381469727
Line speed avg   : 8.944396571866397
Line speed std   : 0.1500571042817685
Statistics, line speed statistics, description and prodRun_time updated successfully.

----------------------------------------------------------------------
ID               : 23592
Prog_Nr          : 8453
Order            : 6284
Production Run   : 2
Stable Start     : 2026-09-17 16:36:19
Stable Stop      : 2026-09-17 18:01:33
Old prodRun_time : 85.23333333333333
Description      : 3114_38,0_41,8_1,90
Calculated prodRun_time: 85.23 minutes
Measurement rows: 2560
Line speed min   : 8.5
Line speed max   : 10.199999809265137
Line speed avg   : 9.320312432199717
Line speed std   : 0.284072216461564
Statistics, line speed statistics, description and prodRun_time updated successfully.

----------------------------------------------------------------------
ID               : 23593
Prog_Nr          : 2960
Order            : 

C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(
C:\Users\velagaturi\AppData\Local\Temp\ipykernel_24712\1290901463.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(
